# Testing `fairscape_models.sql`

In [1]:
#%pip install pydantic sqlalchemy

## Setup 

In [2]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [3]:
try:
	os.remove("integration_test.db")
except:
	pass

In [4]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

from fairscape_models.utils import readCrate
from fairscape_models.sql.models import *
from fairscape_models.sql.ingest import ROCrateIngestRequest
import sqlalchemy as sa
import pathlib

## Load ROCrate Tests

In [5]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [6]:
len(test_rocrate.metadataGraph)

20552

In [7]:
# TODO create a flamegraph of loading rocrate
# py-spy
# pip install py-spy
# py-spy record -o profile.svg -- python myscript.py 
#
# flameprof for cProfile stats
# python -m cProfile -o script.prof myscript.py
# 

In [8]:
# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")


# create table 
Base.metadata.create_all(engine)



In [9]:
# create a session
session = sa.orm.Session(engine) 

ingestRequest = ROCrateIngestRequest(
	model=test_rocrate,
	session=session,
)

In [10]:
# test digest authors
author_data = ingestRequest._digest_authors()



In [11]:
# check authors
author_data, existing_author_ids = ingestRequest._check_authors(author_data)

# if no id's are found author id is an empty dictionary
existing_author_ids

{}

In [12]:
# write the authors
author_ids = ingestRequest._write_authors(author_data)

# join existing author ids to author_ids
full_author_ids = author_ids | existing_author_ids

In [13]:
full_author_ids

{'Abantika Pal': 1,
 'Andrew P. Latham': 2,
 'Robin Bachelder': 3,
 'Steven P. Gygi': 4,
 'Emma Lundberg ': 5,
 'Gege Qian': 6,
 'Kyung-Mee Moon': 7,
 'Joanna Lenkiewicz': 8,
 'Katherine Licon': 9,
 'Xiaoyu Zhao': 10,
 'Laura Pontano Vaites': 11,
 'William Leineweber': 12,
 'Trey Ideker': 13,
 'Ernst Pulido': 14,
 'Neelesh Soni': 15,
 'Dexter Pratt': 16,
 'Leonard J. Foster': 17,
 'Yue Qin': 18,
 'Christopher Churas': 19,
 'Dorothy Tsai': 20,
 'Anthony Cesnik': 21,
 'Peter Zage': 22,
 'J. Wade Harper': 23,
 'Andrej Sali': 24,
 'Mengzhou Hu': 25,
 'Keiichiro Ono': 26,
 'Nicole M. Mattson': 27,
 'Ignacia Echeverria': 28,
 'Aji Palar': 29,
 'Trang Le': 30,
 'Leah V. Schaffer': 31,
 'Jing Chen': 32,
 'Ishan Gaur': 33,
 'Edward L. Huttlin': 34}

In [14]:
author_ids

{'Abantika Pal': 1,
 'Andrew P. Latham': 2,
 'Robin Bachelder': 3,
 'Steven P. Gygi': 4,
 'Emma Lundberg ': 5,
 'Gege Qian': 6,
 'Kyung-Mee Moon': 7,
 'Joanna Lenkiewicz': 8,
 'Katherine Licon': 9,
 'Xiaoyu Zhao': 10,
 'Laura Pontano Vaites': 11,
 'William Leineweber': 12,
 'Trey Ideker': 13,
 'Ernst Pulido': 14,
 'Neelesh Soni': 15,
 'Dexter Pratt': 16,
 'Leonard J. Foster': 17,
 'Yue Qin': 18,
 'Christopher Churas': 19,
 'Dorothy Tsai': 20,
 'Anthony Cesnik': 21,
 'Peter Zage': 22,
 'J. Wade Harper': 23,
 'Andrej Sali': 24,
 'Mengzhou Hu': 25,
 'Keiichiro Ono': 26,
 'Nicole M. Mattson': 27,
 'Ignacia Echeverria': 28,
 'Aji Palar': 29,
 'Trang Le': 30,
 'Leah V. Schaffer': 31,
 'Jing Chen': 32,
 'Ishan Gaur': 33,
 'Edward L. Huttlin': 34}

In [15]:
# TODO slow?
ingestRequest._write_identifier_authors(full_author_ids)

In [16]:
# check that author table has correct author information
session.scalar(sa.select(sa.func.count(AuthorSQL.id)))

34

In [17]:
# check linked authors
session.scalar(sa.select(sa.func.count(AuthorIdentifierSQL.id)))

698633

In [18]:
# digest identifiers
rocrate_identifiers = ingestRequest._digest_identifiers()
ingestRequest._write_identifiers(rocrate_identifiers)

session.flush()
session.commit()

In [19]:
# check the identifiers

In [20]:
# digest all crate elements 
crate_elements = ingestRequest._digest_iterate_elements()

In [21]:
# write rocrate elements
ingestRequest._write_elements()

In [22]:
session.commit()

In [23]:
session.close()

## Test Query

In [24]:
import sys
import os

# load in the fairscape models library
sys.path.insert(0, '/workspaces/fairscape_models')

In [25]:
from fairscape_models.sql.query import QueryByGUID, QueryResponse, SERIALIZE_TYPE
from fairscape_models.utils import readCrate
import sqlalchemy as sa
import pathlib

from fairscape_models.dataset import Dataset
from fairscape_models.software import Software
from fairscape_models.computation import Computation
from fairscape_models.rocrate import ROCrateMetadataElem

# create an engine
engine = sa.create_engine("sqlite:///integration_test.db")

# create table 
# Base.metadata.create_all(engine)


In [26]:
releases = [ elem for elem in pathlib.Path("/mnt/data/Dataverse/").glob("*") if elem.is_dir()]
test_release = pathlib.Path("/mnt/data/Dataverse/U2OS")

test_rocrate_list = list(test_release.glob("*.zip"))
test_rocrate_path = test_release / "cm4ai_u2os_1_ImageDownloader.zip"

test_rocrate = readCrate(test_rocrate_path)

In [27]:
test_crate_metadata = test_rocrate.getCrateMetadata()
test_crate_software = test_rocrate.getSoftware()[0]
test_crate_computation = test_rocrate.getComputations()[0]
test_crate_dataset = test_rocrate.getDatasets()[0]

In [28]:
test_crate_dataset.fileFormat

'.tsv'

In [29]:
session = sa.orm.Session(engine)

In [30]:
def runQuery(GUID: str, session):
	query = QueryByGUID(GUID)
	results = query.execute(session)
	
	return results.transform()


In [31]:
dataset_result = runQuery(test_crate_dataset.guid, session)

#dataset_query = QueryByGUID(test_crate_dataset.guid)
#dataset_query_response = dataset_query.execute(session)
#dataset_query_response.transform()

In [33]:
computation_result = runQuery(test_crate_computation.guid, session)

ValidationError: 2 validation errors for Computation
dateCreated
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
usedSoftware
  Input should be a valid list [type=list_type, input_value='https://fairscape.net/ap...ellmaps-imagedownloader', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type

In [34]:
computation_query = QueryByGUID(test_crate_computation.guid)
computation_query_response = computation_query.execute(session)

In [35]:
computation_query_response.rootEntity

In [36]:
computation_query_response._transform_root_entity()

In [37]:
computation_query_response._convert_metadata()

In [38]:
computation_query_response.metadata

{'name': 'Image Download from Human Protein Atlas',
 'dateCreated': None,
 'datePublished': None,
 'usedSoftware': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader',
 'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [42]:
from fairscape_models.sql.conversion.construct import ConvertComputationToSQL


test_comp = ConvertComputationToSQL(test_crate_computation)

<MetadataTypeEnumSQL.COMPUTATION: 'COMPUTATION'>

In [ ]:
computation_query_response.metadata

{'description': 'Download of immunofluorescence microscopy images from the Human Protein Atlas (HPA v23) for U2OS cell lines using cellmaps_imagedownloader. The tool reads sample metadata and gene lists, then retrieves IF images for each gene target from proteinatlas.org.',
 'usedSoftware': None,
 'name': 'Image Download from Human Protein Atlas',
 'datePublished': None,
 '@id': 'ark:59853/computation-image-download',
 '@type': ['http://www.w3.org/ns/prov#Activity',
  'https://w3id.org/EVI#Computation'],
 'runBy': 'Leah V. Schaffer',
 'dateCreated': None,
 'author': ['Leah V. Schaffer'],
 'keywords': [],
 'hasPart': [],
 'isPartOf': [{'@id': 'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'}]}

In [ ]:
software_result = runQuery(test_crate_software.guid, session)

In [ ]:
crate_result = runQuery(test_crate_metadata.guid, session)

In [ ]:
test_crate_computation.metadataType

['prov:Activity', 'https://w3id.org/EVI#Computation']

In [ ]:
type(computation_result)

NoneType

Dataset(guid='ark:59853/dataset-image-gene-node-attributes-with-locations', metadataType=['prov:Entity', 'https://w3id.org/EVI#Dataset'], name='Image Gene Node Attributes with Locations', isPartOf=[IdentifierValue(guid='ark:59853/rocrate-cm4ai-image-downloader')], author=['Yue Qin', 'Abantika Pal', 'Trang Le', 'Dorothy Tsai', 'Jing Chen', 'Joanna Lenkiewicz', 'J. Wade Harper', 'Laura Pontano Vaites', 'Aji Palar', 'William Leineweber', 'Anthony Cesnik', 'Andrew P. Latham', 'Mengzhou Hu', 'Kyung-Mee Moon', 'Ishan Gaur', 'Andrej Sali', 'Keiichiro Ono', 'Leah V. Schaffer', 'Ignacia Echeverria', 'Leonard J. Foster', 'Steven P. Gygi', 'Emma Lundberg ', 'Christopher Churas', 'Peter Zage', 'Neelesh Soni', 'Gege Qian', 'Trey Ideker', 'Nicole M. Mattson', 'Katherine Licon', 'Robin Bachelder', 'Edward L. Huttlin', 'Ernst Pulido', 'Dexter Pratt', 'Xiaoyu Zhao'], description='Gene node attribute file with subcellular location annotations. Contains 10,825 rows with gene name, Ensembl ID, antibody, i